# Jitter
The aim of this project is to develop a fast sentiment analysis of headlines.  The system is focused on economic jitterness, but can be easily refocused to other topics.

### Architecture
The data pipeline is as follows:

1. Obtain headlines from a list of RSS feeds.
2. Embed headlines (via FinBERT).
3. Score each headline by cosine similarity with preset extreme uncertainty sentences.
4. Average daily scores.

### Demonstration
#### Config
We begin by loading the model. Because we focus on economic sentiment, we will use [FinBERT](https://huggingface.co/ProsusAI/finbert).

In [1]:
from embedding import HFEmbedder

model = "ProsusAI/finbert"
embed = HFEmbedder(model)

dim = embed.dimension()

/home/roberto/Projects/AI/jitter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15812.87it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Next we load the preset centroid sentences that will provide the anchor for sentiment analysis.

In [2]:
import pandas as pd

centroid_sentences = pd.read_csv("centroid.csv")

We compute the embedding for each sentence.

In [3]:
centroid_sentences["embedding"] = centroid_sentences["sentence"].apply(embed)

We compute the centroid vector.

In [4]:
centroid = sum(centroid_sentences.embedding) / len(centroid_sentences.embedding)

**Note.** We could instead compute category-wise centroid vectors.  This would allow scoring for each specific source of jitter.

#### Data retrieval and processing
We download the feed from our feeds list.

In [5]:
from sourcing import from_urls

sources = pd.read_csv("sources.csv")
feed = from_urls(sources.url)

Remove the duplicates.

In [15]:
feed.drop_duplicates(subset="title", inplace=True)

In [16]:
feed

,title,description,timestamp,source,embedding,score
0,Iran War Live Updates: Tehran Says It Is Discu...,The one-page proposal under review would open ...,"(2026, 5, 7, 20, 4, 52, 3, 127, 0)",NYT > World News,"[[tensor(-0.5218), tensor(-0.5200), tensor(-0....",0.459576
1,German Leaders Clash With Spy Chiefs Over Dome...,Intelligence agents have privately warned of t...,"(2026, 5, 7, 9, 0, 26, 3, 127, 0)",NYT > World News,"[[tensor(-0.2542), tensor(-0.5506), tensor(-0....",0.305737
2,Starmer Braces for Local Election Losses Amid ...,Polls predict historic losses for Prime Minist...,"(2026, 5, 7, 13, 48, 17, 3, 127, 0)",NYT > World News,"[[tensor(-0.5373), tensor(0.1633), tensor(0.66...",0.379526
3,"In Hungary, Voters Exposed the Limits of China...","Beijing depended on Hungary’s outgoing leader,...","(2026, 5, 7, 9, 14, 51, 3, 127, 0)",NYT > World News,"[[tensor(-0.2543), tensor(0.0058), tensor(-0.8...",0.419483
4,Israel Says It Killed a Hezbollah Chief Near B...,The strike was the first near the Lebanese cap...,"(2026, 5, 7, 19, 11, 56, 3, 127, 0)",NYT > World News,"[[tensor(-0.1543), tensor(-0.6356), tensor(-0....",0.208031
...,...,...,...,...,...,...
415,"How Running Shoes Have Evolved, From Ancient G...",The race to near-weightlessness has been a dri...,"(2026, 5, 2, 14, 17, 46, 5, 122, 0)",NYT > Technology,"[[tensor(0.4951), tensor(0.3764), tensor(-1.60...",0.219447
416,Pentagon Makes Deals With A.I. Companies to Ex...,The agreements with the technology companies c...,"(2026, 5, 1, 15, 59, 40, 4, 121, 0)",NYT > Technology,"[[tensor(-0.0177), tensor(0.4025), tensor(-0.7...",0.297493
417,OpenAI’s Big Reset + A.I. in the Doctor’s Offi...,Will the rising tide of A.I. adoption lift all...,"(2026, 5, 1, 11, 0, 3, 4, 121, 0)",NYT > Technology,"[[tensor(0.4758), tensor(0.0196), tensor(-0.29...",0.262987
419,Musk vs. Altman: What Is This Really About?,In a landmark trial between Elon Musk and Open...,"(2026, 5, 1, 22, 21, 43, 4, 121, 0)",NYT > Technology,"[[tensor(0.3883), tensor(0.8857), tensor(-1.06...",0.344222


Next, we compute the embedding of each entry.  To do this, we pass `"{title} | {description}"` to our embedding function.

In [17]:
feed["embedding"] = feed.apply(
    lambda x: embed(x["title"] + " | " + x["description"], method="CLS"), axis=1
)

Then we score each embedding using cosine-similarity with respect to the centroid tensor.

In [18]:
from torch.nn.functional import cosine_similarity

feed["score"] = feed["embedding"].apply(
    lambda x: float(cosine_similarity(x, centroid, dim=1))
)

In [19]:
feed

,title,description,timestamp,source,embedding,score
0,Iran War Live Updates: Tehran Says It Is Discu...,The one-page proposal under review would open ...,"(2026, 5, 7, 20, 4, 52, 3, 127, 0)",NYT > World News,"[[tensor(-0.5218), tensor(-0.5200), tensor(-0....",0.459576
1,German Leaders Clash With Spy Chiefs Over Dome...,Intelligence agents have privately warned of t...,"(2026, 5, 7, 9, 0, 26, 3, 127, 0)",NYT > World News,"[[tensor(-0.2542), tensor(-0.5506), tensor(-0....",0.305737
2,Starmer Braces for Local Election Losses Amid ...,Polls predict historic losses for Prime Minist...,"(2026, 5, 7, 13, 48, 17, 3, 127, 0)",NYT > World News,"[[tensor(-0.5373), tensor(0.1633), tensor(0.66...",0.379526
3,"In Hungary, Voters Exposed the Limits of China...","Beijing depended on Hungary’s outgoing leader,...","(2026, 5, 7, 9, 14, 51, 3, 127, 0)",NYT > World News,"[[tensor(-0.2543), tensor(0.0058), tensor(-0.8...",0.419483
4,Israel Says It Killed a Hezbollah Chief Near B...,The strike was the first near the Lebanese cap...,"(2026, 5, 7, 19, 11, 56, 3, 127, 0)",NYT > World News,"[[tensor(-0.1543), tensor(-0.6356), tensor(-0....",0.208031
...,...,...,...,...,...,...
415,"How Running Shoes Have Evolved, From Ancient G...",The race to near-weightlessness has been a dri...,"(2026, 5, 2, 14, 17, 46, 5, 122, 0)",NYT > Technology,"[[tensor(0.4951), tensor(0.3764), tensor(-1.60...",0.219447
416,Pentagon Makes Deals With A.I. Companies to Ex...,The agreements with the technology companies c...,"(2026, 5, 1, 15, 59, 40, 4, 121, 0)",NYT > Technology,"[[tensor(-0.0177), tensor(0.4025), tensor(-0.7...",0.297493
417,OpenAI’s Big Reset + A.I. in the Doctor’s Offi...,Will the rising tide of A.I. adoption lift all...,"(2026, 5, 1, 11, 0, 3, 4, 121, 0)",NYT > Technology,"[[tensor(0.4758), tensor(0.0196), tensor(-0.29...",0.262987
419,Musk vs. Altman: What Is This Really About?,In a landmark trial between Elon Musk and Open...,"(2026, 5, 1, 22, 21, 43, 4, 121, 0)",NYT > Technology,"[[tensor(0.3883), tensor(0.8857), tensor(-1.06...",0.344222


In [21]:
feed.score.median()

np.float64(0.30609217286109924)